In [2]:
!pip install -q mediapipe imagehash tqdm

## 1. Download and unzip the dataset

In [3]:
import os, zipfile, requests
from tqdm import tqdm

DATA_DIR = '/content/data'
ZIP_PATH = '/content/ualpha40.zip'
URL = 'https://data.mendeley.com/public-api/zip/3pvnnckxyb/download/1'

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0))
        with open(ZIP_PATH, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
print('Zip size (GB):', round(os.path.getsize(ZIP_PATH) / 1e9, 2))

100%|██████████| 5.80G/5.80G [03:16<00:00, 29.5MB/s]

Zip size (GB): 5.8


**Fallback:** if the download fails, download the zip manually from  
https://data.mendeley.com/datasets/3pvnnckxyb/1 (*Download All*), upload it once to Google Drive, then use the cell below and set `ZIP_PATH` to the Drive path.

In [ ]:
# FALLBACK ONLY: uncomment if the direct download above failed
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_PATH = '/content/drive/MyDrive/ualpha40.zip'

In [4]:
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f.lower().endswith('.zip'):
            p = os.path.join(root, f)
            with zipfile.ZipFile(p) as z:
                z.extractall(os.path.join(root, f[:-4]))
            print('Extracted nested zip:', p)

## 2. Folder structure

In [5]:
def show_tree(path, max_depth=3, max_items=8, prefix='', depth=0):
    if depth > max_depth:
        return
    entries = sorted(os.listdir(path))
    dirs = [e for e in entries if os.path.isdir(os.path.join(path, e))]
    files = [e for e in entries if not os.path.isdir(os.path.join(path, e))]
    for d in dirs[:max_items]:
        print(f'{prefix}[DIR] {d}/  ({len(os.listdir(os.path.join(path, d)))} items)')
        show_tree(os.path.join(path, d), max_depth, max_items, prefix + '    ', depth + 1)
    if len(dirs) > max_items:
        print(f'{prefix}... and {len(dirs) - max_items} more folders')
    for f in files[:3]:
        print(f'{prefix}{f}')
    if len(files) > 3:
        print(f'{prefix}... and {len(files) - 3} more files')

show_tree(DATA_DIR)

[DIR] Pakistan Sign Language Urdu Alphabets/  (40 items)
    [DIR] 1-Hay/  (659 items)
        s0251-10hay-Generated-169.jpg
        s0251-10hay-Generated-170.jpg
        s0251-10hay-Generated-171.jpg
        ... and 656 more files
    [DIR] 2-Hay/  (87 items)
        s0251.mp4
        s0252.mp4
        s0253.mp4
        ... and 84 more files
    [DIR] Ain/  (586 items)
        s0251-25ain-Generated-100.jpg
        s0251-25ain-Generated-101.jpg
        s0251-25ain-Generated-102.jpg
        ... and 583 more files
    [DIR] Alif/  (660 items)
        s0251-02alif-Generated-1.jpg
        s0251-02alif-Generated-10.jpg
        s0251-02alif-Generated-11.jpg
        ... and 657 more files
    [DIR] Alifmad/  (105 items)
        s0252.mp4
        s0253.mp4
        s0254.mp4
        ... and 102 more files
    [DIR] Aray/  (102 items)
        s0251.mp4
        s0252.mp4
        s0253.mp4
        ... and 99 more files
    [DIR] Bay/  (690 items)
        s0251-03bay-Generated-25.jpg
        s0251-

## 3. File inventory (type, count, size)

In [6]:
import pandas as pd

rows = []
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        p = os.path.join(root, f)
        rows.append({
            'path': p,
            'rel_folder': os.path.relpath(root, DATA_DIR),
            'filename': f,
            'ext': os.path.splitext(f)[1].lower(),
            'size_kb': os.path.getsize(p) / 1024,
        })
inv = pd.DataFrame(rows)
print('Total files:', len(inv))
inv.groupby('ext').agg(files=('path', 'count'), total_mb=('size_kb', lambda s: round(s.sum() / 1024, 1)))

Total files: 23258


,files,total_mb
ext,,
.jpg,22865,5166.7
.mp4,393,543.0


In [7]:
pd.set_option('display.max_rows', 200)
inv.groupby(['rel_folder', 'ext']).size().reset_index(name='count')

,rel_folder,ext,count
0,Pakistan Sign Language Urdu Alphabets/1-Hay,.jpg,659
1,Pakistan Sign Language Urdu Alphabets/2-Hay,.mp4,87
2,Pakistan Sign Language Urdu Alphabets/Ain,.jpg,586
3,Pakistan Sign Language Urdu Alphabets/Alif,.jpg,660
4,Pakistan Sign Language Urdu Alphabets/Alifmad,.mp4,105
5,Pakistan Sign Language Urdu Alphabets/Aray,.mp4,102
6,Pakistan Sign Language Urdu Alphabets/Bay,.jpg,690
7,Pakistan Sign Language Urdu Alphabets/Byeh,.jpg,562
8,Pakistan Sign Language Urdu Alphabets/Chay,.jpg,628
9,Pakistan Sign Language Urdu Alphabets/Cyeh,.jpg,562


## 4. Quick checks against what the dataset page / proposal claims

In [8]:
img_ext, vid_ext = ['.jpg', '.jpeg', '.png'], ['.mp4', '.avi', '.mov']
print(f"Images found: {inv['ext'].isin(img_ext).sum()}   (proposal expects ~22,280)")
print(f"Videos found: {inv['ext'].isin(vid_ext).sum()}   (proposal expects ~393)")
print('Other file types:', inv.loc[~inv['ext'].isin(img_ext + vid_ext), 'ext'].value_counts().to_dict())

Images found: 22865   (proposal expects ~22,280)
Videos found: 393   (proposal expects ~393)
Other file types: {}


## 5. Save the inventory

In [9]:
os.makedirs('/content/outputs', exist_ok=True)
inv.drop(columns='path').to_csv('/content/outputs/file_inventory.csv', index=False)
print('Saved file_inventory.csv -> download it and commit it to dataset/ in the repo')

Saved file_inventory.csv -> download it and commit it to dataset/ in the repo


## Findings from this step
*(Fill in after running)*
- Folder layout:
- How labels are stored (folder name / file name):
- Image count vs expected:
- Video count vs expected:
- Anything unexpected (extra files, nested folders, odd extensions):